Project 1: Finding similar items
The task is to implement a detector of pairs of similar abstract for scientific papers, using the abstract field in the arXiv dataset, published under the Public Domain license. You can choose how to encode abstracts, provided that similarity is measured using the Jaccard similarity and that at least one of the techniques studied in the course is applied.



## Data Source

Download the arXiv dataset from Kaggle:
- [arXiv Metadata Dataset on Kaggle](https://www.kaggle.com/datasets/Cornell-University/arxiv)

Use the Kaggle API to download:
```python
kaggle datasets download -d cornell-university/arxiv
```
```

In [1]:
import os
import json
import random
from compression import zlib
from dotenv import load_dotenv
import kaggle
import re
from pyspark.sql import SparkSession


In [2]:
load_dotenv()  
os.environ["KAGGLE_USERNAME"]  
os.environ["KAGGLE_KEY"]     

'KGAT_68e13934bb524e1efed42de6a7bc30c7'

In [3]:
CONFIG = {
    "DATA_PATH": "data/arxiv-metadata-oai-snapshot.json",  # produced by your kaggle download
    "SAMPLE_FRACTION": 0.08,       # fraction of the dataset to sample for testing         
    "SHINGLE_SIZE": 3,            # k-word shingles
    "NUM_HASHES": 100,            # MinHash permutation functions
    "NUM_BANDS": 20,              # LSH bands (must divide NUM_HASHES evenly)
    "JACCARD_THRESHOLD": 0.3,
    "RANDOM_SEED": 42,
}
assert CONFIG["NUM_HASHES"] % CONFIG["NUM_BANDS"] == 0, (
    f"NUM_HASHES ({CONFIG['NUM_HASHES']}) must be evenly divisible by "
    f"NUM_BANDS ({CONFIG['NUM_BANDS']})"
)

ROWS_PER_BAND = CONFIG["NUM_HASHES"] // CONFIG["NUM_BANDS"]
MERSENNE_PRIME = (1 << 61) - 1 

STOPWORDS = {
    "the","a","an","and","or","of","in","on","to","for","is","are","was",
    "were","be","been","being","this","that","these","those","with","as",
    "by","at","from","it","its","we","our","which","also","can","such",
    "using","based","paper","results","show","study","propose","present",
    "between","into","than","then","not","no","have","has","had","will",
    "would","may","could","should","their","they","them","he","she","his",
    "her","i","you","your","if","but","so","up","out","about","over",
    "under","new","two","one","three","used","method","methods","approach",
    "data","however","where","when","what","how","proposed","novel","framework",
    "experiments","experimental","demonstrate","achieve","performance","improve",
    "improved","compared","existing","different","various","several",
}

## Dataset loading and parsing

In [4]:
#os.system('kaggle datasets download -d cornell-university/arxiv --unzip -p data/')


In [5]:
def create_spark_session(app_name="ArxivSimilarAbstracts"):
    spark = (
        SparkSession.builder
        .appName(app_name)
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.shuffle.partitions", "16")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark

In [6]:
def load_sample_rdd(sc, path, sample_fraction): 
    """RDD[str] one raw JSON line per document, sampled distributedly."""
    return sc.textFile(path).sample( #python list[str] of str, each str is a line of json text
        withReplacement=False, 
        fraction=sample_fraction, 
        seed=CONFIG["RANDOM_SEED"]
    )#each line is a str which itself contains json text


def parse_record(line):
    """
    JSON line -> (paper_id, title, abstract) or None if malformed/too short.
    """
    try:
        #taking one element per row of json string and execute per row
        rec = json.loads(line) #deserialize JSON line into a dictionary 
        paper_id = rec.get("id")
        title = (rec.get("title") or "").strip() #had a bug wih crushing if no title in spark
        abstract = (rec.get("abstract") or "").strip()
        if not paper_id or len(abstract) < 20:
            return None
        return (paper_id, title, abstract)
    except Exception:
        return None 

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize(text):
    return text.split() #spliting into words


def remove_stopwords(tokens): #filtering out stopwords and words with length <= 2
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]


def get_shingles(tokens, k):
    """k-word shingles, e.g. ['a','b','c','d'], k=3 -> {'a b c', 'b c d'}"""
    if len(tokens) < k:
        return {" ".join(tokens)} if tokens else set()
    return {" ".join(tokens[i:i + k]) for i in range(len(tokens) - k + 1)}

#executed on each partition of the RDD, so it can be parallelized across the cluster
def build_shingle_rdd(raw_rdd, k):
    """RDD[str] raw lines -> RDD[(paper_id, title, shingle_set)]"""
    parsed = raw_rdd.map(parse_record).filter(lambda x: x is not None)

    def process(record):
        paper_id, title, abstract = record
        tokens = remove_stopwords(tokenize(clean_text(abstract)))
        shingles = get_shingles(tokens, k)
        return (paper_id, title, shingles)

    return parsed.map(process).filter(lambda x: len(x[2]) > 0)


## Jacard simiairty core algorithm

In [7]:
def jaccard_similarity(set_a, set_b):
    """|A ∩ B| / |A ∪ B|"""
    if not set_a and not set_b:
        return 0.0
    intersection = len(set_a & set_b)  # |A ∩ B|
    union = len(set_a | set_b)  # |A ∪ B|
    if union == 0:
        return 0.0
    return intersection / union


## MinHash core algorithm

Its a tennuique to compress large vraiable setshingles into a small fixet size signatures.
P(signature values match) ≈ true Jaccard similarity of the original sets


In [8]:
def make_hash_coeffs(num_hashes=CONFIG["NUM_HASHES"], seed=CONFIG["RANDOM_SEED"],prime=MERSENNE_PRIME):
    """Instead of shuffling each shingle, we can use hash functions to simulate the effect of shuffling. 
        Each hash function is defined by a pair of coefficients (a, b) for the linear hash function 
        h(x) = (a * x + b) % prime, very large"""
    rng = random.Random(seed)
    return [(rng.randint(1, prime - 1), # pick a random 'a'  (1 to prime-1)
             rng.randint(0, prime - 1))  #pick a random 'b'  (0 to prime-1)
            for _ in range(num_hashes)]


#CRC32 ~4.3 billion distinct values
def stable_hash(shingle):
    """Deterministic across executors; 32-bit. Built-in hash() is NOT stable"""
    return zlib.crc32(shingle.encode("utf-8"))          


def minhash_signature(shingles, coeffs, prime=MERSENNE_PRIME):
    if not shingles:
        return tuple([prime] * len(coeffs)) #for empty shingles, return a tuple of prime values as a sentinel        
    ints = [stable_hash(s) for s in shingles] #converting each shingle string to an integer
    return tuple(min((a * x + b) % prime for x in ints) for a, b in coeffs) #tupple(hashable) of 100ints - a fixed-length 100-number signature


def build_signature_rdd(shingle_rdd, coeffs):
    """RDD[(paper_id, title, shingles)] -> RDD[(paper_id, title,shingles, signature)]"""
    return shingle_rdd.map(lambda x: (x[0], x[1], x[2], minhash_signature(x[2], coeffs))) #keeping shingles for Jacard on candidate shingle sets


def estimated_jaccard_from_signatures(sig_a, sig_b):
    """Fraction of matching positions ≈ true Jaccard similarity"""
    matches = sum(1 for x, y in zip(sig_a, sig_b) if x == y)
    return matches / len(sig_a)


## LSH core algorithm
to turn 157k signatures into a short list of candidate pairs without comparing all 1.2×10¹⁰ combinations

At this point i have:
1. Shingles - 3word phrases
2. MinHAsh signatures of 100ints

So I will implement a band - lice of the signature to compare each agains other docs
1. Cut papers to band by MinHash signature
2. Bucket the bands - grouping identicals
3. Finding candidate pair
4. Ingnore the rest

In [9]:
def split_signature_into_bands(signature, num_bands, rows_per_band):
    """One signature -> [(band_index, band_tuple), ...]"""
    bands = []
    for b in range(num_bands):
        start = b * rows_per_band
        bands.append((b, tuple(signature[start:start + rows_per_band])))
    return bands

def emit_band_keys(sig_rdd, num_bands, rows):
    """Each doc -> one (band_key, paper_id) row per band"""
    def band_emit(record):
        paper_id, title, shingles, sig = record
        for band_key in split_signature_into_bands(sig, num_bands, rows):
            yield (band_key, paper_id)
    return sig_rdd.flatMap(band_emit)

def band_key_emit(record): #making only 20 rows(bands) per bucket
    """One document -> (band_key, paper_id) for each of its bands"""
    paper_id, title, shingles, sig = record      # all 4 fields
    for band_key in split_signature_into_bands(sig, CONFIG["NUM_BANDS"], ROWS_PER_BAND):
        yield (band_key, paper_id)


def bucket_bands(band_rows): #band_rows gonna call in main section by  band_rows = sig_rdd.flatMap()
    """ Group all paper_ids that share a band_key for further comparison. Returns RDD[(band_key, [paper_id1, paper_id2, ...])]"""
    return (band_rows
            .groupByKey()          # (band_key, <iterable of ids>)
            .mapValues(list)       # (band_key, [id1, id2, ...])
            .filter(lambda kv: len(kv[1]) > 1))   # drop singleton buckets (no pairs)

def candidate_pairs_from_bucket(ids): #now comapre lements inside a bucket
    """All unique list of id's iterated on (i, j) inside one bucket"""
    ids = sorted(set(ids))
    return [(ids[i], ids[j])
            for i in range(len(ids))
            for j in range(i + 1, len(ids))]

# Main

In [10]:
def inspect_pipeline(raw_rdd, shingle_rdd, sig_rdd):
    """Sample data + distributions"""
    example = raw_rdd.filter(lambda l: parse_record(l) is not None).take(1)[0]
    print("\n Loaded data example: ", example[:200], ", parsed: ", parse_record(example))

    print("\nCheck first 5 shingle docs: ")
    for pid, title, sh in shingle_rdd.take(5):
        print(f"{pid}{title[:70]} "); print(f"{len(sh)} shingles, e.g. {list(sh)[:3]}.")
        st = shingle_rdd.map(lambda x: len(x[2])).stats()
    print("\nStatistcs for shingles: ")
    print(f"min={int(st.min())} mean={st.mean():.1f} max={int(st.max())}")

    print("\nCheck first 2 signatures:")
    for pid, title, sh, sig in sig_rdd.take(2):
        print(f"For {pid} with title {title[:60]}: shingles={len(sh)} with example signatures: {sig[:5]}, len {len(sig)}")


def validate_minhash(sig_rdd, config, n_pairs=500, sample_size=200):
    """Empirically check MinHash estimates Jaccard within ~1/sqrt(k)."""
    import random
    sample = sig_rdd.take(sample_size)
    rng = random.Random(config["RANDOM_SEED"])

    errors = []
    for _ in range(n_pairs):
        a, b = rng.sample(sample, 2)
        true = jaccard_similarity(a[2], b[2])
        est  = estimated_jaccard_from_signatures(a[3], b[3])
        errors.append(abs(true - est))

    mae = sum(errors) / len(errors)
    theory = 1 / (config["NUM_HASHES"] ** 0.5)
    print("\nDoes MinHash actually estimate Jaccard?")
    print(f"MinHash validation found a self-similarity of {estimated_jaccard_from_signatures(sample[0][3], sample[0][3]):.3f}, a mean absolute error of {mae:.4f}, and a theoretical error of approximately {theory:.4f} based on 1/sqrt(k).")


def verify_candidates(candidates, sig_rdd, threshold, top_n):
    """Exact Jaccard on candidate pairs; keep + rank those above threshold."""
    needed_ids = set(candidates.flatMap(lambda pair: pair).collect())
    info = (sig_rdd.filter(lambda r: r[0] in needed_ids)
                   .map(lambda r: (r[0], (r[1], r[2])))
                   .collectAsMap())

    def score_pair(pair):
        id_a, id_b = pair
        if id_a not in info or id_b not in info:          # info is a dict -> no .value
            return None
        title_a, shingles_a = info[id_a]
        title_b, shingles_b = info[id_b]
        return (id_a, title_a, id_b, title_b, jaccard_similarity(shingles_a, shingles_b))

    results = (candidates
               .map(score_pair)
               .filter(lambda x: x is not None and x[4] >= threshold)     # <- the PARAM
               .cache())

    total = candidates.count()                                            # <- local, not a global
    print(f"\nSimilar pairs within threshold {threshold}:")
    print(f"Confirmed: {results.count()} of {total} candidates")
    for id_a, title_a, id_b, title_b, sim in results.takeOrdered(top_n, key=lambda x: -x[4]):  # <- the PARAM
        print(f"  {sim:.3f}  {id_a} <-> {id_b}")
        print(f"          {title_a[:60]}")
        print(f"          {title_b[:60]}")
    return results



In [11]:
if __name__ == "__main__":
    spark = create_spark_session(app_name="ArxivSimilarAbstracts")
    sc = spark.sparkContext

    # 1. Load and sample
    raw_rdd = load_sample_rdd(sc, CONFIG["DATA_PATH"], CONFIG["SAMPLE_FRACTION"]).cache()
    print("1. Data loading and sampling")
    print(f"   Raw lines sampled: {raw_rdd.count():,}")

    # 2. Shingling
    shingle_rdd = build_shingle_rdd(raw_rdd, CONFIG["SHINGLE_SIZE"]).cache()
    n = shingle_rdd.count()
    print("\n2. Shingling")
    print(f"   Docs kept: {n:,}")

    # 3. MinHash signatures
    HASH_COEFFS = make_hash_coeffs()
    sig_rdd = build_signature_rdd(shingle_rdd, HASH_COEFFS).cache()
    print("\n3. MinHash signatures")
    print(f"   Signatures built: {sig_rdd.count():,} (length {CONFIG['NUM_HASHES']})")

    # 4. LSH candidate pairs
    band_rows  = sig_rdd.flatMap(band_key_emit)
    buckets    = bucket_bands(band_rows)
    candidates = (buckets
                  .flatMap(lambda kv: candidate_pairs_from_bucket(kv[1]))
                  .distinct()
                  .cache())
    num_candidates = candidates.count()
    total_possible = n * (n - 1) // 2
    print("\n4. LSH candidate pairs")
    print(f"   Bands: {CONFIG['NUM_BANDS']} x {ROWS_PER_BAND} rows")
    print(f"   Candidate pairs: {num_candidates:,} of {total_possible:,} "
          f"({num_candidates / total_possible:.6%})")


      #5. Jaccard similarity
    results = verify_candidates(candidates, sig_rdd, CONFIG["JACCARD_THRESHOLD"], top_n=20)
    spark.stop()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/11 20:22:31 WARN Utils: Your hostname, Ayalas-Mac-mini.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.10 instead (on interface en1)
26/09/11 20:22:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 20:22:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


1. Data loading and sampling


   Raw lines sampled: 252,917



2. Shingling
   Docs kept: 252,911

3. MinHash signatures


   Signatures built: 252,911 (length 100)



4. LSH candidate pairs
   Bands: 20 x 5 rows
   Candidate pairs: 282 of 31,981,860,505 (0.000001%)



Similar pairs within threshold 0.3:


Confirmed: 276 of 282 candidates
  1.000  astro-ph/0302340 <-> quant-ph/0102068
          A new map of neutrino cosmology - revised bounds on the numb
          Faster Database search in Quantum Computing
  1.000  quant-ph/0503178 <-> quant-ph/9711026
          Withdrawn
          Secure Multi-Party Addition via Quantum States
  1.000  astro-ph/0505508 <-> quant-ph/0211125
          Cosmic Rays VIII
          Degree of entanglement for bipartite pure states
  1.000  astro-ph/0504599 <-> gr-qc/0101076
          An application of Green-function methods to gravitational ra
          On the Green functions of gravitational radiation theory
  1.000  hep-th/9911044 <-> hep-th/9911046
          Quantum Nonsymmetric Gravity and The Superfiber Bundle Forma
          Symmetries of Quantum Nonsymmetric Gravity
  1.000  1103.5311 <-> astro-ph/0302340
          Possible resolution to the conflict between the principle of
          A new map of neutrino cosmology - revised bounds on the numb
  1.000